In [ ]:
import marimo as mo

#Определение цели и постановка задачи
    Цель: определение эмоциональной окраски твита

    Задача: применение методов классического ML для определения позитивной/негативной окраски
    небольшого фрагмента текста на основе его содержания и времени написания.

    Задача будет решаться путем бинарном классификации, а в качестве метрики выбираем log-loss, accuracy и f1-меру

#Анализ данных

    На этом этапе оцениваем структуру данных, качество признаков, распределение метки и определение
    потенциальных проблем перед обучением моделей

In [ ]:
from copy import deepcopy
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

#Работа с датасетом

Рассмотрим датасет и определим спектр работ созданию наборов данных

In [ ]:
data=pd.read_csv('data/raw/training_data.csv', delimiter = ',', quotechar='"', names = ['y','id message','date','flag','user','text'],encoding = 'latin1')
data.head()

,y,id message,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there."


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   y           1600000 non-null  int64 
 1   id message  1600000 non-null  int64 
 2   date        1600000 non-null  object
 3   flag        1600000 non-null  object
 4   user        1600000 non-null  object
 5   text        1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


Пропусков данных нет

Датасет состоит из следующих столбцов:

1) y (таргет) - числовой признак принимает значения полярности сообщения (0 - негативный, 4 - позитивный)

2) id message - числовой признак уникальный ID сообщения

3) date - категориальный признак дата и время сообщения

4) flag - категориальный признак запроса

5) user - категориальный признак (ник пользователя)

6) text - категориальный признак (сообщение)

Следующий шаг посмотреть распределение таргета и постановка задачи в рамках pipeline

In [ ]:
data['y'].value_counts() #данные разделились ровно 50 на 50, что позволяет использовать accuracy и f1-меру

,count
y,
0,800000
4,800000


In [ ]:
data['flag'].value_counts() #бесполезный столбец

,count
flag,
NO_QUERY,1600000


Удаляем ненужные колонки, дорабатываем значения и подготовливаем датасет к обучению

In [ ]:
clean_data=deepcopy(data)
clean_data['y']=clean_data['y'].map({0:0, 4:1}) #меняем 4 на 1
clean_data['date']=clean_data['date'].str.replace(r'[A-Z]{3}',' ', regex=True) #убрали метку региона
clean_data['date']=pd.to_datetime(clean_data['date'], format= '%a %b %d %H:%M:%S %Y') #переводим дату и время в более удобный формат
clean_data['day']=clean_data['date'].dt.strftime('%a') #в отдельный стоблец выносим день недели
clean_data['hour']=clean_data['date'].dt.hour
def get_time(hour):
    if 6<=hour<11:
        return "morning"
    elif 11<=hour<16:
        return "day"
    elif 16<=21:
        return "evening"
    else:
        return "night"
clean_data['time']=clean_data['hour'].apply(get_time) #заменили время на промежутки времени (утро, день, вечер и ночь)
clean_data['text']=clean_data['text'].apply(lambda x: re.sub(r'(@\w+|http\S+)', '', x).strip()) #очистим текст от артефактов пользователей
def clean_text(text):
    # нижний регистр
    text = text.lower()
    # убрать все спецсимволы, оставляем только буквы и пробелы
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # убрать лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text
clean_data['text']=clean_data['text'].apply(clean_text)
clean_data.drop(['id message','flag','user','hour','date'], axis=1, inplace=True)

clean_data.head()

,y,text,day,time
0,0,awww thats a bummer you shoulda got david carr of third day to do it d,Mon,evening
1,0,is upset that he cant update his facebook by texting it and might cry as a result school today also blah,Mon,evening
2,0,i dived many times for the ball managed to save 50 the rest go out of bounds,Mon,evening
3,0,my whole body feels itchy and like its on fire,Mon,evening
4,0,no its not behaving at all im mad why am i here because i cant see you all over there,Mon,evening


#Итого

Результатом этого этапа является:

- сформулированная задача (применение методов классического ML для определения позитивной/негативной окраски)
- определены метрики модели (accuracy, f1-мера)
- данные очищены, не содержат пропусков
- распределение классов равномерно

На следующих этапах будет построена baseline модель классификации

In [ ]:
clean_data.to_csv('data/processed/clean_data.csv', index=False, encoding='utf-8')